# Mohr-Coulomb Validation — Embedded Crack with NCP Contact

Biaxial compression on a unit square with an embedded (non-boundary-intersecting) crack.
The background stress state is carried entirely by **s0/w0 offsets** — the solver computes
only the perturbation.  Two test cases validate the NCP formulation against the classical
Mohr-Coulomb criterion:

1. **Locked** (θ < θ_crit): perturbation is zero, crack stays closed.
2. **Sliding** (θ > θ_crit): NCP releases, crack slides.

## 1. Imports

In [1]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '4')

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from skfem.models.elasticity import lame_parameters

from poroelasticity import CGPoroelastostatics, CrackMeshBuilder
from poroelasticity.cg_dashboard import CGPoroelasticDashboard

import solve_nivp
from solve_nivp.ncp_contact import build_dynamic_ncp_contact

print(f"solve_nivp version: {getattr(solve_nivp, '__version__', 'dev')}")
print('Imports OK')

solve_nivp version: dev
Imports OK


## 2. Configuration

`ALPHA_BIOT = 0` decouples fluid and solid — pure elastodynamics.  High bulk viscous
damping (`BULK_MU_V`, `BULK_LAM_V`) overdamps the system so the Radau IIA time-stepping
is effectively quasi-static.  The crack is embedded at `(0.5, 0.5)` with length 0.4,
well inside the unit square.

In [2]:
# --- Material (Biot off, overdamped) ---
NU          = 0.25
G_SHEAR     = 22.0e3               # MPa
E_YOUNG     = 2.0 * G_SHEAR * (1.0 + NU)
ALPHA_BIOT  = 0.0
BETA_FLUID  = 8.5e-5
ETA_FLUID   = 2.0e-18 / 3600.0
K_PERM      = 1.0e-15 * 1.0e-6
DENSITY     = 1.0
BULK_MU_V   = 1.0e6                # large shear viscosity → overdamped
BULK_LAM_V  = 1.0e6                # large bulk viscosity → overdamped
SCALE_L     = 1.0
SCALE_EPS   = 1.0e-3

# --- Domain ---
XMIN, XMAX = 0.0, 1.0
YMIN, YMAX = 0.0, 1.0
N_ELEM     = 20

# --- Crack geometry ---
CRACK_X0     = 0.5
CRACK_Y0     = 0.5
CRACK_LENGTH = 0.4

# --- Loading (compression positive, MPa) ---
SIGMA_RIGHT = 10.0
SIGMA_TOP   = 30.0

# --- Friction ---
MU = 0.6

# --- Time stepping ---
TMAX    = 5.0
H_FIXED = 0.5

# --- Derived ---
LAM, MU_LAME = lame_parameters(E_YOUNG, NU)
PARAMS = (MU_LAME, LAM, ALPHA_BIOT, BETA_FLUID, K_PERM / ETA_FLUID)

# Principal stresses (compression positive)
S1 = max(SIGMA_RIGHT, SIGMA_TOP)   # σ₁ = 30 (vertical)
S3 = min(SIGMA_RIGHT, SIGMA_TOP)   # σ₃ = 10 (horizontal)

print(f"Domain:  [{XMIN},{XMAX}] × [{YMIN},{YMAX}],  N_ELEM = {N_ELEM}")
print(f"Crack:   centre=({CRACK_X0},{CRACK_Y0}), length={CRACK_LENGTH}")
print(f"Loading: σ_right={SIGMA_RIGHT}, σ_top={SIGMA_TOP}  →  σ₁={S1}, σ₃={S3}")
print(f"Friction: μ = {MU}")
print(f"Damping: bulk_mu_v={BULK_MU_V:.1e}, bulk_lam_v={BULK_LAM_V:.1e}")
print(f"Time:    tmax={TMAX}, h={H_FIXED}")

Domain:  [0.0,1.0] × [0.0,1.0],  N_ELEM = 20
Crack:   centre=(0.5,0.5), length=0.4
Loading: σ_right=10.0, σ_top=30.0  →  σ₁=30.0, σ₃=10.0
Friction: μ = 0.6
Damping: bulk_mu_v=1.0e+06, bulk_lam_v=1.0e+06
Time:    tmax=5.0, h=0.5
